# Load Steerling-8B-Instruct and run a prompt

This notebook loads the model and tokenizer directly with `from_pretrained`, then uses Steerling's native masked-diffusion generator. It does not use a Transformers pipeline. Expect roughly 18 GB of model weights and use a CUDA GPU with at least 24 GB VRAM.

## Bind installs and subprocesses to this notebook kernel

A running notebook cannot replace its own interpreter. This block makes shell commands follow the already-selected Jupyter kernel. `%pip` always installs into that kernel, including hosted Conda or system-backed kernels.

In [ ]:
import os
import sys
from pathlib import Path

kernel_prefix = Path(sys.prefix).resolve()
base_prefix = Path(sys.base_prefix).resolve()
kernel_bin = Path(sys.executable).resolve().parent
os.environ["PATH"] = os.pathsep.join(
    [str(kernel_bin), *os.environ.get("PATH", "").split(os.pathsep)]
)
if kernel_prefix != base_prefix:
    os.environ["VIRTUAL_ENV"] = str(kernel_prefix)
    os.environ["PIP_REQUIRE_VIRTUALENV"] = "true"
else:
    os.environ.pop("VIRTUAL_ENV", None)
    os.environ.pop("PIP_REQUIRE_VIRTUALENV", None)
print("Active notebook environment:", kernel_prefix)
print("Python executable:", sys.executable)
print("Environment type:", "venv" if kernel_prefix != base_prefix else "hosted/base kernel")

In [ ]:
%pip install -q "accelerate>=1.2,<2" "transformers>=4.48,<5"
%pip install -q --ignore-requires-python "steerling @ git+https://github.com/guidelabs/steerling.git@f34ffa89e46969445f3cf6e7c885e9623a2047c1"

In [ ]:
import torch
from steerling import GenerationConfig, SteerlingGenerator
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "guidelabs/steerling-8b-instruct"
MODEL_REVISION = "6e5a87d00d45348001810c30fe9bd65110b69fc2"

print("Python:", sys.version)
if sys.version_info < (3, 13):
    print("Note: Steerling's Python >=3.13 package metadata was bypassed for this kernel.")

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU with at least 24 GB VRAM is required.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))

## Direct model loading

`AutoModel.from_pretrained` instantiates the repository's custom causal-diffusion model. The immutable revision keeps the executable remote model code and weights reproducible.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map={"": "cuda"},
)
model.eval()

generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")
print(type(model))
print(generator)
print(f"Interpretable: {generator.is_interpretable}")
print(f"Diffusion block size: {generator.diff_block_size}")

## Format and run one prompt

In [ ]:
user_prompt = "Explain how a diffusion language model differs from an autoregressive language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": user_prompt},
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(formatted_prompt)

In [ ]:
generation_config = GenerationConfig(
    max_new_tokens=128,
    steps=128,
    seed=42,
    stop_tokens=[tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

with torch.inference_mode():
    response = generator.generate(formatted_prompt, generation_config)

print("Prompt:", user_prompt)
print("Response:", response)